# Acceptance: fused weighted reductions and XLA:CPU settings

**This is a sign-off page, not a tutorial.** It exercises `plans/fuse-weighted-reduce.plan.md`. A rate
that sums many weighted `Reduce` terms, `sum_k c_k * Reduce(where=sel_k)`, now compiles to one gather and
dot product, and on import summer4 picks faster XLA:CPU settings for the installed jaxlib. Nothing here
asserts; the automated checks live in `tests/test_fuse_reductions.py` and `tests/test_cpu_runtime.py`.
The guide page *What summer4 sets up for you*, section 5, describes the runtime settings.

In [ ]:
import sys
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

import summer4

BENCH = Path("..") / ".." / "benchmarks"
if not (BENCH / "weighted_reduce.py").exists():
    BENCH = Path("benchmarks")
sys.path.insert(0, str(BENCH.resolve()))
import weighted_reduce as wr

## A. What summer4 chose for XLA:CPU

On jaxlib 0.4.32 to 0.6.x summer4 selects XLA's legacy CPU runtime; on 0.7.0 to 0.12.x it turns off
the thunk runtime's concurrency-optimised scheduler. The table shows this process.

In [ ]:
info = asdict(summer4.runtime_info())
pd.Series({k: info[k] for k in ("cpu_runtime", "cpu_runtime_source", "cpu_concurrency_scheduler")}, name="this process").to_frame()

> **What to check**
>
> - With jaxlib 0.6.x: `cpu_runtime` is `legacy`, source `default`, scheduler empty (`None`).
> - With jaxlib 0.7 to 0.12: `cpu_runtime` is `thunk`, source `jaxlib`, `cpu_concurrency_scheduler` is `False`.
> - If you started Jupyter with `SUMMER4_CPU_RUNTIME=thunk` or your own `XLA_FLAGS`, the source names it.

## B. The same model, fused and term by term

The benchmark model (`benchmarks/weighted_reduce.py`) is a 41-compartment chain whose infectious pool
weights each of 36 grid cells with its own `Reduce` term. It is compiled twice: with the default
(`fuse_reductions=True`) and with `fuse_reductions=False`. The table counts the vector field's jaxpr
equations and gives the largest relative difference between the two in the vector field, the final
prevalence after 2000 Euler steps, and its gradient with respect to two parameters.

In [ ]:
fused = wr.make_fns("terms", "euler")
plain = wr.make_fns("terms", "euler", compile_kw={"fuse_reductions": False})


def rel(a, b):
    x, y = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    scale = np.maximum(np.abs(x), np.abs(y))
    return float(np.max(np.where(scale > 0, np.abs(x - y) / np.where(scale > 0, scale, 1), 0.0)))


theta = fused["theta"]
vf = [
    m["cm"].vector_field(0.0, m["y0"], m["cm"].prepare(dict(wr.PARAMS))) for m in (fused, plain)
]
pd.DataFrame(
    {
        "fused": [*wr.vf_eqns(fused["cm"], fused["y0"]), None],
        "term by term": [*wr.vf_eqns(plain["cm"], plain["y0"]), None],
        "relative difference": [
            rel(*vf),
            rel(fused["solve"](theta), plain["solve"](theta)),
            rel(fused["grad"](theta), plain["grad"](theta)),
        ],
    },
    index=["vector-field equations / vector field", "nested equations / prevalence", "gradient"],
)

> **What to check**
>
> - The fused column has far fewer equations (84 against 263 top level and 407 nested on jaxlib 0.6.2).
> - Every relative difference is at rounding level, below about `1e-13`: fusion changes the order of a sum, not the answer.

## C. Timing on this computer

Best of 10 calls after compiling, for the forward solve and the gradient. These are timed here, so they
depend on the computer running this notebook; `benchmarks/README.md` has the numbers measured on the Mac
mini with `devq bench --strict`.

In [ ]:
times = pd.DataFrame(
    {
        name: {what: wr.best_time(m[what], theta, 10) * 1e3 for what in ("solve", "grad")}
        for name, m in (("fused", fused), ("term by term", plain))
    }
).T
times.plot.bar(
    barmode="group",
    title="2000 Euler steps: ms per call",
    labels={"index": "", "value": "ms", "variable": ""},
).show()
times.round(2)

> **What to check**
>
> - The fused gradient bar is clearly lower than the term-by-term one.
> - The fused solve bar is no higher than the term-by-term one (with the default runtime settings of section A).